In [ ]:

# ============================================================
# CELL 1: DEPENDENCIES AND IMPORTS
# ============================================================

# Install these if they are not already available:
#
# %pip install ipywidgets networkx ipycytoscape

import os
import re
import json
import ipaddress
import subprocess
import shutil
import uuid
import threading
from pathlib import Path
from datetime import datetime

import ipywidgets as widgets
import networkx as nx

from IPython.display import display, HTML, clear_output

try:
    from ipycytoscape import CytoscapeWidget
    CYTOSCAPE_AVAILABLE = True
except ImportError:
    CytoscapeWidget = None
    CYTOSCAPE_AVAILABLE = False

print("Network Namespace Builder initialized")
print("Cytoscape available:", CYTOSCAPE_AVAILABLE)
print("Python:", os.sys.version)


In [ ]:

# ============================================================
# CELL 2: APPLICATION SETTINGS
# ============================================================

CONFIG_VERSION = 2

# Safety defaults
DRY_RUN = True

ALLOW_PHYSICAL_MOVEMENT = False

# Explicit physical interface allowlist.
# Empty means no physical interface can be moved.
PHYSICAL_INTERFACE_ALLOWLIST = set()

COMMAND_TIMEOUT = 60

# WireGuard configuration files must exist for real deployment.
# Disabled WireGuard entries are imported and visualized but skipped.
REQUIRE_WIREGUARD_FILES = True

# Application-owned resources.
OWNERSHIP_FILE = Path(
    "/var/tmp/network_namespace_builder_ownership.json"
)

# Internal deployment state.
created_namespaces = set()
created_physical = []
created_wireguard = []

deployment_active = False
deployment_lock = threading.Lock()

# Configuration sections.
EMPTY_CONFIG = {
    "version": CONFIG_VERSION,
    "namespaces": [],
    "links": [],
    "switches": [],
    "physical": [],
    "wireguard": [],
    "ips": [],
    "routes": [],
    "rules": [],
    "forwarding": []
}

config = json.loads(json.dumps(EMPTY_CONFIG))

print("Configuration schema version:", CONFIG_VERSION)
print("Dry run:", DRY_RUN)
print("Physical interface movement:", ALLOW_PHYSICAL_MOVEMENT)


In [ ]:

# ============================================================
# CELL 3: COMMAND EXECUTION AND LOGGING
# ============================================================

output_log = widgets.Output()

deployment_messages = []


def log(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    formatted = f"[{timestamp}] {message}"

    deployment_messages.append(formatted)

    with output_log:
        print(formatted)


def clear_log():
    deployment_messages.clear()

    with output_log:
        clear_output()


def execute(command, namespace=None, check=True):
    """
    Execute a command safely.

    namespace:
        Optional Linux network namespace.

    DRY_RUN:
        Logs the command without executing it.
    """

    if not isinstance(command, list) or not command:
        raise ValueError("Command must be a non-empty list")

    command = [str(x) for x in command]

    if namespace:
        command = [
            "ip", "netns", "exec",
            namespace
        ] + command

    printable = " ".join(command)

    if DRY_RUN:
        log(f"[DRY RUN] {printable}")
        return True

    log(f"EXEC: {printable}")

    try:
        result = subprocess.run(
            command,
            check=False,
            capture_output=True,
            text=True,
            timeout=COMMAND_TIMEOUT
        )

        if result.stdout.strip():
            log(result.stdout.strip())

        if result.stderr.strip():
            log(result.stderr.strip())

        if check and result.returncode != 0:
            raise RuntimeError(
                f"Command failed ({result.returncode}): "
                f"{printable}\n{result.stderr}"
            )

        return result.returncode == 0

    except subprocess.TimeoutExpired:
        raise RuntimeError(
            f"Command timed out: {printable}"
        )


def command_exists(name):
    return shutil.which(name) is not None


def require_commands():
    required = ["ip", "sysctl"]

    missing = [
        command for command in required
        if not command_exists(command)
    ]

    if missing:
        raise RuntimeError(
            "Missing required commands: "
            + ", ".join(missing)
        )

    if config["wireguard"] and any(
        item.get("enabled", True)
        for item in config["wireguard"]
    ):
        for command in ("wg",):
            if not command_exists(command):
                raise RuntimeError(
                    f"Missing WireGuard command: {command}"
                )

    log("Required system commands available")


In [ ]:

# ============================================================
# CELL 4: CONFIGURATION VALIDATION
# ============================================================

NAME_PATTERN = re.compile(r"^[a-zA-Z0-9_.-]{1,15}$")


def validate_name(value, description="Name"):
    if not isinstance(value, str):
        raise ValueError(f"{description} must be a string")

    value = value.strip()

    if not NAME_PATTERN.fullmatch(value):
        raise ValueError(
            f"Invalid {description}: {value!r}"
        )

    return value


def validate_table(value):
    if value == "main":
        return "main"

    if isinstance(value, bool):
        raise ValueError("Invalid routing table")

    try:
        value = int(value)
    except (ValueError, TypeError):
        raise ValueError("Routing table must be numeric or main")

    if not 1 <= value <= 4294967295:
        raise ValueError("Routing table ID out of range")

    return value


def validate_ip(item):
    address = ipaddress.ip_interface(item["addr"])

    expected = 4 if item["version"] == "IPv4" else 6

    if address.version != expected:
        raise ValueError(
            f"Address family mismatch: {item['addr']}"
        )

    return address


def validate_route(route):
    validate_name(route["ns"], "Route namespace")

    destination = ipaddress.ip_network(
        route["dst"],
        strict=False
    )

    family = route.get(
        "family",
        f"IPv{destination.version}"
    )

    if family != f"IPv{destination.version}":
        raise ValueError(
            "Route family does not match destination"
        )

    if route.get("via"):
        gateway = ipaddress.ip_address(route["via"])

        if gateway.version != destination.version:
            raise ValueError(
                "Route gateway and destination families differ"
            )

    if route.get("dev"):
        validate_name(route["dev"], "Route interface")

    if not route.get("via") and not route.get("dev"):
        raise ValueError(
            "Route requires a gateway or interface"
        )

    if route.get("metric") is not None:
        if (
            isinstance(route["metric"], bool)
            or not isinstance(route["metric"], int)
            or route["metric"] < 0
        ):
            raise ValueError("Invalid route metric")

    validate_table(route.get("table", "main"))

    return True


def validate_rule(rule):
    validate_name(rule["ns"], "Rule namespace")

    priority = rule.get("priority")

    if (
        isinstance(priority, bool)
        or not isinstance(priority, int)
        or not 1 <= priority <= 32765
    ):
        raise ValueError("Invalid policy rule priority")

    validate_table(rule["table"])

    if rule.get("from"):
        ipaddress.ip_network(
            rule["from"],
            strict=False
        )

    if rule.get("to"):
        ipaddress.ip_network(
            rule["to"],
            strict=False
        )

    if not rule.get("from") and not rule.get("to"):
        raise ValueError(
            "Policy rule needs a source or destination selector"
        )

    family = rule.get("family", "IPv4")

    if family not in ("IPv4", "IPv6"):
        raise ValueError("Invalid policy rule family")

    for field in ("from", "to"):
        if rule.get(field):
            address = ipaddress.ip_network(
                rule[field],
                strict=False
            )

            expected = 4 if family == "IPv4" else 6

            if address.version != expected:
                raise ValueError(
                    f"Policy rule {field} family mismatch"
                )

    return True


def validate_config(data):
    if not isinstance(data, dict):
        raise ValueError("Configuration must be an object")

    if data.get("version") != CONFIG_VERSION:
        raise ValueError(
            f"Unsupported configuration version: "
            f"{data.get('version')}"
        )

    required = set(EMPTY_CONFIG)

    missing = required - set(data)

    if missing:
        raise ValueError(
            f"Missing configuration sections: {sorted(missing)}"
        )

    for section in required - {"version"}:
        if not isinstance(data[section], list):
            raise ValueError(
                f"Configuration section {section} must be a list"
            )

    namespaces = set()

    for ns in data["namespaces"]:
        validate_name(ns, "Namespace")

        if ns in namespaces:
            raise ValueError(f"Duplicate namespace: {ns}")

        namespaces.add(ns)

    interfaces = set()
    host_veth_names = set()

    # Veth pairs.
    for link in data["links"]:
        for side in ("A", "B"):
            ns = link[f"ns{side}"]

            if ns not in namespaces:
                raise ValueError(
                    f"Unknown namespace in link: {ns}"
                )

            host_name = validate_name(
                link[f"int{side}"],
                "Veth host name"
            )

            if host_name in host_veth_names:
                raise ValueError(
                    f"Duplicate veth host name: {host_name}"
                )

            host_veth_names.add(host_name)

            name = validate_name(
                link.get(f"rename{side}") or host_name,
                "Veth interface name"
            )

            key = (ns, name)

            if key in interfaces:
                raise ValueError(
                    f"Duplicate interface: {ns}/{name}"
                )

            interfaces.add(key)

    # Physical interfaces.
    physical_host_names = set()

    for item in data["physical"]:
        ns = item["ns"]

        if ns not in namespaces:
            raise ValueError(
                f"Unknown physical interface namespace: {ns}"
            )

        host_iface = validate_name(
            item["iface"],
            "Physical host interface"
        )

        if host_iface in physical_host_names:
            raise ValueError(
                f"Physical interface specified more than once: "
                f"{host_iface}"
            )

        physical_host_names.add(host_iface)

        rename = validate_name(
            item.get("rename") or host_iface,
            "Physical interface rename"
        )

        key = (ns, rename)

        if key in interfaces:
            raise ValueError(
                f"Duplicate interface: {ns}/{rename}"
            )

        interfaces.add(key)

    # Switches.
    bridges = set()
    bridge_ports = set()

    for switch in data["switches"]:
        ns = switch["ns"]

        if ns not in namespaces:
            raise ValueError(
                f"Unknown switch namespace: {ns}"
            )

        bridge = validate_name(
            switch["bridge"],
            "Bridge name"
        )

        if (ns, bridge) in interfaces:
            raise ValueError(
                f"Bridge name conflicts with interface: {bridge}"
            )

        if (ns, bridge) in bridges:
            raise ValueError(
                f"Duplicate bridge: {ns}/{bridge}"
            )

        bridges.add((ns, bridge))

        ports = switch.get("ports", [])

        if len(ports) != len(set(ports)):
            raise ValueError(
                f"Duplicate bridge ports in {ns}/{bridge}"
            )

        for port in ports:
            validate_name(port, "Bridge port")

            key = (ns, port)

            if key not in interfaces:
                raise ValueError(
                    f"Unknown bridge port: {ns}/{port}"
                )

            if key in bridge_ports:
                raise ValueError(
                    f"Interface assigned to multiple bridges: "
                    f"{ns}/{port}"
                )

            bridge_ports.add(key)

    # IP assignments.
    ip_assignments = set()

    for item in data["ips"]:
        ns = item["ns"]
        iface = item["iface"]

        if ns not in namespaces:
            raise ValueError(
                f"Unknown IP namespace: {ns}"
            )

        if (ns, iface) not in interfaces:
            raise ValueError(
                f"IP assigned to unknown interface: {ns}/{iface}"
            )

        address = validate_ip(item)

        key = (ns, iface, str(address))

        if key in ip_assignments:
            raise ValueError(
                f"Duplicate IP assignment: {key}"
            )

        ip_assignments.add(key)

        if (ns, iface) in bridge_ports:
            raise ValueError(
                f"Do not assign an IP to bridge port "
                f"{ns}/{iface}; assign it to the bridge instead"
            )

    # WireGuard.
    wireguard_interfaces = set()

    for item in data["wireguard"]:
        ns = item["ns"]

        if ns not in namespaces:
            raise ValueError(
                f"Unknown WireGuard namespace: {ns}"
            )

        iface = validate_name(
            item["iface"],
            "WireGuard interface"
        )

        key = (ns, iface)

        if key in interfaces or key in wireguard_interfaces:
            raise ValueError(
                f"Duplicate WireGuard interface: {ns}/{iface}"
            )

        wireguard_interfaces.add(key)

        path = Path(item["config"])

        if not path.is_absolute():
            raise ValueError(
                "WireGuard configuration path must be absolute"
            )

        if path.name != iface + ".conf":
            raise ValueError(
                f"WireGuard config must be named {iface}.conf"
            )

        if item.get("enabled", True) and REQUIRE_WIREGUARD_FILES:
            if not DRY_RUN and not path.is_file():
                raise FileNotFoundError(
                    f"WireGuard configuration not found: {path}"
                )

    all_interfaces = interfaces | wireguard_interfaces

    # Routes.
    for route in data["routes"]:
        validate_route(route)

        if route["ns"] not in namespaces:
            raise ValueError("Unknown route namespace")

        if route.get("dev"):
            if (route["ns"], route["dev"]) not in all_interfaces:
                raise ValueError(
                    f"Route references unknown interface: "
                    f"{route['ns']}/{route['dev']}"
                )

    # Rules.
    rule_priorities = set()

    for rule in data["rules"]:
        validate_rule(rule)

        if rule["ns"] not in namespaces:
            raise ValueError("Unknown policy rule namespace")

        key = (
            rule["ns"],
            rule.get("family", "IPv4"),
            rule["priority"]
        )

        if key in rule_priorities:
            raise ValueError(
                f"Duplicate policy rule priority: {key}"
            )

        rule_priorities.add(key)

    # Forwarding.
    forwarding_namespaces = set()

    for item in data["forwarding"]:
        ns = item["ns"]

        if ns not in namespaces:
            raise ValueError(
                f"Unknown forwarding namespace: {ns}"
            )

        if ns in forwarding_namespaces:
            raise ValueError(
                f"Duplicate forwarding configuration: {ns}"
            )

        forwarding_namespaces.add(ns)

        for family in ("ipv4", "ipv6"):
            if family in item and not isinstance(
                item[family], bool
            ):
                raise ValueError(
                    f"{family} forwarding must be boolean"
                )

    log("Configuration validation successful")

    return True


In [ ]:

# ============================================================
# CELL 5: TOPOLOGY VISUALIZATION
# ============================================================

class NetworkBuilderApp:

    def __init__(self):
        self.G = nx.MultiGraph()

        self.cytoscape_widget = None

        if CYTOSCAPE_AVAILABLE:
            self.cytoscape_widget = CytoscapeWidget()
            self.cytoscape_widget.set_style([
                {
                    "selector": "node",
                    "style": {
                        "label": "data(label)",
                        "background-color": "#2b5797",
                        "color": "#ffffff",
                        "text-wrap": "wrap",
                        "font-size": "12px",
                        "shape": "round-rectangle",
                        "padding": "12px",
                        "width": "label",
                        "height": "label"
                    }
                },
                {
                    "selector": ".bridge",
                    "style": {
                        "background-color": "#26734d",
                        "shape": "diamond"
                    }
                },
                {
                    "selector": ".wireguard",
                    "style": {
                        "background-color": "#7952a3"
                    }
                },
                {
                    "selector": "edge",
                    "style": {
                        "label": "data(label)",
                        "font-size": "9px",
                        "line-color": "#999999",
                        "width": 2,
                        "text-background-color": "#ffffff",
                        "text-background-opacity": 0.8
                    }
                }
            ])

    def update_graph(self, data=None):
        if data is None:
            data = config

        self.G.clear()

        for ns in data["namespaces"]:
            self.G.add_node(
                ns,
                label=f"{ns}\n(Namespace)",
                kind="namespace"
            )

        for link in data["links"]:
            ns_a = link["nsA"]
            ns_b = link["nsB"]

            name_a = link.get("renameA") or link["intA"]
            name_b = link.get("renameB") or link["intB"]

            self.G.add_edge(
                ns_a,
                ns_b,
                label=f"{name_a} <-> {name_b}"
            )

        for switch in data["switches"]:
            ns = switch["ns"]
            bridge = switch["bridge"]

            bridge_id = f"{ns}:{bridge}"

            self.G.add_node(
                bridge_id,
                label=f"{bridge}\n(Bridge)",
                kind="bridge"
            )

            self.G.add_edge(
                ns,
                bridge_id,
                label="Bridge"
            )

            for port in switch["ports"]:
                self.G.add_edge(
                    bridge_id,
                    f"{ns}/{port}",
                    label=port
                )

        for item in data["wireguard"]:
            if not item.get("enabled", True):
                continue

            ns = item["ns"]
            iface = item["iface"]

            wg_id = f"{ns}/{iface}"

            self.G.add_node(
                wg_id,
                label=f"{iface}\n(WireGuard)",
                kind="wireguard"
            )

            self.G.add_edge(
                ns,
                wg_id,
                label="Tunnel"
            )

        for item in data["physical"]:
            ns = item["ns"]
            iface = item.get("rename") or item["iface"]

            host_id = f"HOST:{item['iface']}"

            self.G.add_node(
                host_id,
                label=f"{item['iface']}\n(Host NIC)"
            )

            self.G.add_edge(
                host_id,
                ns,
                label=iface
            )

        if self.cytoscape_widget:
            self.cytoscape_widget.graph.clear()
            self.cytoscape_widget.graph.add_graph_from_networkx(
                self.G
            )


app = NetworkBuilderApp()
app.update_graph()

print("Topology visualization initialized")


In [ ]:

# ============================================================
# CELL 6: LINUX NETWORK RESOURCE OPERATIONS
# ============================================================

def save_ownership():
    data = {
        "namespaces": sorted(created_namespaces),
        "physical": created_physical,
        "wireguard": created_wireguard
    }

    if DRY_RUN:
        return

    OWNERSHIP_FILE.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp = OWNERSHIP_FILE.with_suffix(".tmp")

    temp.write_text(
        json.dumps(data, indent=2)
    )

    os.replace(temp, OWNERSHIP_FILE)


def load_ownership():
    if not OWNERSHIP_FILE.exists():
        return {
            "namespaces": [],
            "physical": [],
            "wireguard": []
        }

    return json.loads(
        OWNERSHIP_FILE.read_text()
    )


def namespace_exists(ns):
    if DRY_RUN:
        return ns in created_namespaces

    result = subprocess.run(
        ["ip", "netns", "list"],
        capture_output=True,
        text=True,
        check=True
    )

    existing = {
        line.split()[0]
        for line in result.stdout.splitlines()
        if line.split()
    }

    return ns in existing


def create_namespaces(data):
    global created_namespaces

    for ns in data["namespaces"]:

        if namespace_exists(ns):
            if ns not in created_namespaces:
                raise RuntimeError(
                    f"Namespace already exists and is not owned "
                    f"by this application: {ns}"
                )

            log(f"Namespace already owned: {ns}")

        else:
            execute(["ip", "netns", "add", ns])

            created_namespaces.add(ns)

        # Bring up loopback.
        execute(
            ["ip", "link", "set", "lo", "up"],
            namespace=ns
        )

    save_ownership()


def create_links(data):
    for link in data["links"]:

        int_a = link["intA"]
        int_b = link["intB"]

        ns_a = link["nsA"]
        ns_b = link["nsB"]

        rename_a = link.get("renameA") or int_a
        rename_b = link.get("renameB") or int_b

        log(
            f"Creating veth: "
            f"{ns_a}/{rename_a} <-> {ns_b}/{rename_b}"
        )

        execute([
            "ip", "link", "add",
            int_a,
            "type", "veth",
            "peer", "name", int_b
        ])

        execute([
            "ip", "link", "set",
            int_a, "netns", ns_a
        ])

        execute([
            "ip", "link", "set",
            int_b, "netns", ns_b
        ])

        if rename_a != int_a:
            execute([
                "ip", "link", "set",
                int_a, "name", rename_a
            ], namespace=ns_a)

        if rename_b != int_b:
            execute([
                "ip", "link", "set",
                int_b, "name", rename_b
            ], namespace=ns_b)

        execute([
            "ip", "link", "set",
            rename_a, "up"
        ], namespace=ns_a)

        execute([
            "ip", "link", "set",
            rename_b, "up"
        ], namespace=ns_b)


def create_switches(data):
    for switch in data["switches"]:

        ns = switch["ns"]
        bridge = switch["bridge"]

        log(f"Creating bridge {ns}/{bridge}")

        execute([
            "ip", "link", "add",
            bridge, "type", "bridge"
        ], namespace=ns)

        stp = int(bool(switch.get("stp", False)))
        vlan = int(bool(switch.get("vlan_filtering", False)))

        execute([
            "ip", "link", "set",
            bridge, "type", "bridge",
            "stp_state", str(stp),
            "vlan_filtering", str(vlan)
        ], namespace=ns)

        execute([
            "ip", "link", "set",
            bridge, "up"
        ], namespace=ns)

        for port in switch["ports"]:

            execute([
                "ip", "link", "set",
                port, "master", bridge
            ], namespace=ns)

            execute([
                "ip", "link", "set",
                port, "up"
            ], namespace=ns)


def move_physical_interfaces(data):
    global created_physical

    if not data["physical"]:
        log("No physical interfaces configured")
        return

    if not ALLOW_PHYSICAL_MOVEMENT:
        raise RuntimeError(
            "Physical interface movement is disabled. "
            "Enable it explicitly before deployment."
        )

    for item in data["physical"]:

        host_iface = item["iface"]
        ns = item["ns"]
        rename = item.get("rename") or host_iface

        if host_iface not in PHYSICAL_INTERFACE_ALLOWLIST:
            raise RuntimeError(
                f"Physical interface is not allowlisted: "
                f"{host_iface}"
            )

        if not DRY_RUN:
            result = subprocess.run(
                ["ip", "link", "show", "dev", host_iface],
                capture_output=True
            )

            if result.returncode != 0:
                raise RuntimeError(
                    f"Host interface does not exist: {host_iface}"
                )

        log(
            f"Moving host interface {host_iface} "
            f"into namespace {ns}"
        )

        execute([
            "ip", "link", "set",
            host_iface, "down"
        ])

        execute([
            "ip", "link", "set",
            host_iface, "netns", ns
        ])

        if rename != host_iface:
            execute([
                "ip", "link", "set",
                host_iface, "name", rename
            ], namespace=ns)

        execute([
            "ip", "link", "set",
            rename, "up"
        ], namespace=ns)

        created_physical.append({
            "host_iface": host_iface,
            "ns": ns,
            "namespace_iface": rename
        })

    save_ownership()


def configure_ips(data):
    for item in data["ips"]:

        ns = item["ns"]
        iface = item["iface"]
        addr = item["addr"]

        family = (
            "-4" if item["version"] == "IPv4"
            else "-6"
        )

        log(f"Assigning {addr} to {ns}/{iface}")

        execute([
            "ip", family, "address",
            "add", addr,
            "dev", iface
        ], namespace=ns)

        execute([
            "ip", "link", "set",
            iface, "up"
        ], namespace=ns)


In [ ]:

# ============================================================
# CELL 7: WIREGUARD AND LAYER 3 CONFIGURATION
# ============================================================

def configure_wireguard(data):
    global created_wireguard

    for item in data["wireguard"]:

        if not item.get("enabled", True):
            log(
                f"WireGuard disabled: "
                f"{item['ns']}/{item['iface']}"
            )
            continue

        ns = item["ns"]
        iface = item["iface"]
        path = Path(item["config"])

        if not DRY_RUN:
            if not path.is_file():
                raise FileNotFoundError(
                    f"WireGuard configuration not found: {path}"
                )

            if not os.access(path, os.R_OK):
                raise PermissionError(
                    f"Cannot read WireGuard configuration: {path}"
                )

        log(f"Creating WireGuard interface {ns}/{iface}")

        execute([
            "ip", "link", "add",
            iface, "type", "wireguard"
        ], namespace=ns)

        execute([
            "wg", "setconf",
            iface, str(path)
        ], namespace=ns)

        execute([
            "ip", "link", "set",
            iface, "up"
        ], namespace=ns)

        created_wireguard.append({
            "ns": ns,
            "iface": iface
        })

    save_ownership()


def configure_forwarding(data):
    for item in data["forwarding"]:

        ns = item["ns"]

        ipv4 = int(item.get("ipv4", False))
        ipv6 = int(item.get("ipv6", False))

        log(f"Configuring forwarding in {ns}")

        execute([
            "sysctl", "-w",
            f"net.ipv4.ip_forward={ipv4}"
        ], namespace=ns)

        execute([
            "sysctl", "-w",
            f"net.ipv6.conf.all.forwarding={ipv6}"
        ], namespace=ns)


def configure_routes(data):
    for route in data["routes"]:

        ns = route["ns"]

        network = ipaddress.ip_network(
            route["dst"],
            strict=False
        )

        family = (
            "-4" if network.version == 4
            else "-6"
        )

        command = [
            "ip", family, "route", "replace",
            str(network)
        ]

        if route.get("via"):
            command.extend([
                "via", route["via"]
            ])

        if route.get("dev"):
            command.extend([
                "dev", route["dev"]
            ])

        table = validate_table(
            route.get("table", "main")
        )

        if table != "main":
            command.extend([
                "table", str(table)
            ])

        if route.get("metric") is not None:
            command.extend([
                "metric", str(route["metric"])
            ])

        if route.get("onlink", False):
            command.append("onlink")

        log(
            f"Installing route in {ns}: "
            f"{route['dst']}"
        )

        execute(command, namespace=ns)


def configure_rules(data):
    for rule in data["rules"]:

        ns = rule["ns"]

        family = (
            "-4" if rule.get("family", "IPv4") == "IPv4"
            else "-6"
        )

        command = [
            "ip", family, "rule", "add",
            "priority", str(rule["priority"])
        ]

        if rule.get("from"):
            command.extend([
                "from", rule["from"]
            ])

        if rule.get("to"):
            command.extend([
                "to", rule["to"]
            ])

        command.extend([
            "table", str(validate_table(rule["table"]))
        ])

        log(
            f"Installing policy rule in {ns}: "
            f"priority {rule['priority']}"
        )

        execute(command, namespace=ns)


def verify_deployment(data):
    """
    Basic post-deployment checks.
    """

    if DRY_RUN:
        log("Verification skipped during dry-run")
        return

    for ns in data["namespaces"]:

        if not namespace_exists(ns):
            raise RuntimeError(
                f"Namespace missing after deployment: {ns}"
            )

        execute([
            "ip", "link", "show"
        ], namespace=ns)

        execute([
            "ip", "-4", "route", "show"
        ], namespace=ns)

        execute([
            "ip", "-6", "route", "show"
        ], namespace=ns)

    for item in data["wireguard"]:
        if item.get("enabled", True):
            execute([
                "wg", "show", item["iface"]
            ], namespace=item["ns"])

    log("Basic deployment verification completed")


In [ ]:

# ============================================================
# CELL 8: DEPLOYMENT CONTROLLER AND CLEANUP
# ============================================================

# Correct dry-run namespace handling.
def namespace_exists(ns):
    if DRY_RUN:
        return False

    result = subprocess.run(
        ["ip", "netns", "list"],
        capture_output=True,
        text=True,
        check=True
    )

    existing = {
        line.split()[0]
        for line in result.stdout.splitlines()
        if line.split()
    }

    return ns in existing


# Replace namespace creation with dry-run-safe handling.
def create_namespaces(data):
    for ns in data["namespaces"]:

        if namespace_exists(ns):
            raise RuntimeError(
                f"Refusing to adopt existing namespace: {ns}"
            )

        execute(["ip", "netns", "add", ns])

        if not DRY_RUN:
            created_namespaces.add(ns)

        execute(
            ["ip", "link", "set", "lo", "up"],
            namespace=ns
        )

    save_ownership()


def deploy_configuration(data=None):
    global deployment_active
    global created_namespaces
    global created_physical
    global created_wireguard

    if not deployment_lock.acquire(blocking=False):
        raise RuntimeError(
            "Another deployment operation is running"
        )

    try:
        if deployment_active:
            raise RuntimeError("Deployment already active")

        deployment_active = True

        if data is None:
            data = config

        clear_log()

        # A fresh deployment session.
        created_namespaces = set()
        created_physical = []
        created_wireguard = []

        log("Starting configuration validation")

        validate_config(data)

        if not DRY_RUN:
            if os.geteuid() != 0:
                raise PermissionError(
                    "Real deployment requires root privileges"
                )

            require_commands()

            if OWNERSHIP_FILE.exists():
                previous = load_ownership()

                if previous.get("namespaces"):
                    raise RuntimeError(
                        "An existing deployment ownership manifest "
                        "was found. Clean it up before deploying "
                        "a new topology."
                    )

        log("Beginning deployment")

        create_namespaces(data)

        create_links(data)

        move_physical_interfaces(data)

        create_switches(data)

        configure_ips(data)

        configure_wireguard(data)

        configure_forwarding(data)

        configure_routes(data)

        configure_rules(data)

        if not DRY_RUN:
            save_ownership()

            verify_deployment(data)

        log("Deployment completed successfully")

    except Exception as error:
        log(f"DEPLOYMENT FAILED: {error}")

        if not DRY_RUN:
            log("Attempting rollback")
            cleanup_configuration(silent=True)

        raise

    finally:
        deployment_active = False
        deployment_lock.release()


def restore_physical_interfaces(physical_items):
    for item in reversed(physical_items):

        ns = item["ns"]
        iface = item["namespace_iface"]
        host_iface = item["host_iface"]

        log(
            f"Restoring physical interface "
            f"{host_iface} from {ns}"
        )

        # Move interface back into the host namespace.
        execute([
            "ip", "link", "set",
            iface, "netns", "1"
        ], namespace=ns)

        if iface != host_iface:
            execute([
                "ip", "link", "set",
                iface, "name", host_iface
            ])

        execute([
            "ip", "link", "set",
            host_iface, "up"
        ])


def cleanup_configuration(silent=False):
    global created_namespaces
    global created_physical
    global created_wireguard

    if DRY_RUN:
        log("Cleanup preview:")

        for ns in sorted(config["namespaces"]):
            execute(["ip", "netns", "delete", ns])

        log("Dry-run cleanup preview completed")
        return

    if not silent:
        log("Starting cleanup")

    # Recover ownership after a kernel/network restart or
    # notebook kernel restart.
    if not created_namespaces:
        ownership = load_ownership()

        created_namespaces = set(
            ownership.get("namespaces", [])
        )

        created_physical = ownership.get(
            "physical", []
        )

        created_wireguard = ownership.get(
            "wireguard", []
        )

    # Restore physical interfaces before deleting namespaces.
    for item in reversed(created_physical):

        ns = item["ns"]

        if not namespace_exists(ns):
            log(
                f"Cannot restore {item['host_iface']}: "
                f"namespace {ns} no longer exists"
            )
            continue

        restore_physical_interfaces([item])

    # Delete only namespaces recorded as owned.
    for ns in sorted(created_namespaces):

        if namespace_exists(ns):
            log(f"Deleting owned namespace: {ns}")

            execute([
                "ip", "netns", "delete", ns
            ])

    created_namespaces.clear()
    created_physical.clear()
    created_wireguard.clear()

    if OWNERSHIP_FILE.exists():
        OWNERSHIP_FILE.unlink()

    log("Cleanup completed")


# Convenience wrappers.
def apply():
    deploy_configuration(config)


def cleanup():
    cleanup_configuration()


In [ ]:

# ============================================================
# CELL 9: GUI INPUTS AND EVENT HANDLERS
# ============================================================

# ------------------------------------------------------------
# NAMESPACE
# ------------------------------------------------------------

ns_input = widgets.Text(
    description="Namespace:",
    placeholder="client1"
)

btn_add_node = widgets.Button(
    description="Add Namespace",
    button_style="info"
)


def handle_add_node(button):
    try:
        ns = validate_name(ns_input.value, "Namespace")

        if ns in config["namespaces"]:
            raise ValueError("Namespace already exists")

        config["namespaces"].append(ns)

        ns_input.value = ""

        refresh_topology()

        log(f"Namespace added: {ns}")

    except Exception as error:
        log(f"ERROR: {error}")


# ------------------------------------------------------------
# VETH LINKS
# ------------------------------------------------------------

link_name1 = widgets.Text(
    description="Veth A:",
    value="veth-a"
)

link_ns1 = widgets.Text(
    description="Namespace A:"
)

link_name1_rename = widgets.Text(
    description="Rename A:",
    value="eth0"
)

link_name2 = widgets.Text(
    description="Veth B:",
    value="veth-b"
)

link_ns2 = widgets.Text(
    description="Namespace B:"
)

link_name2_rename = widgets.Text(
    description="Rename B:",
    value="eth0"
)

btn_add_link = widgets.Button(
    description="Add Veth Link",
    button_style="info"
)


def handle_add_link(button):
    try:
        link = {
            "intA": link_name1.value.strip(),
            "nsA": link_ns1.value.strip(),
            "renameA": link_name1_rename.value.strip(),
            "intB": link_name2.value.strip(),
            "nsB": link_ns2.value.strip(),
            "renameB": link_name2_rename.value.strip()
        }

        candidate = json.loads(json.dumps(config))
        candidate["links"].append(link)

        validate_config(candidate)

        config["links"].append(link)

        refresh_topology()

        log("Veth link added")

    except Exception as error:
        log(f"LINK ERROR: {error}")


# ------------------------------------------------------------
# SWITCHES / BRIDGES
# ------------------------------------------------------------

switch_ns = widgets.Text(
    description="Namespace:"
)

switch_bridge = widgets.Text(
    description="Bridge:",
    value="br0"
)

switch_ports = widgets.Text(
    description="Ports:",
    placeholder="eth1,eth2,eth3"
)

switch_stp = widgets.Checkbox(
    description="STP",
    value=False
)

switch_vlan = widgets.Checkbox(
    description="VLAN filtering",
    value=False
)

btn_add_switch = widgets.Button(
    description="Add Switch",
    button_style="info"
)


def handle_add_switch(button):
    try:
        ports = [
            p.strip()
            for p in switch_ports.value.split(",")
            if p.strip()
        ]

        switch = {
            "ns": switch_ns.value.strip(),
            "bridge": switch_bridge.value.strip(),
            "ports": ports,
            "stp": switch_stp.value,
            "vlan_filtering": switch_vlan.value
        }

        candidate = json.loads(json.dumps(config))
        candidate["switches"].append(switch)

        validate_config(candidate)

        config["switches"].append(switch)

        refresh_topology()

        log(f"Switch added: {switch['ns']}/{switch['bridge']}")

    except Exception as error:
        log(f"SWITCH ERROR: {error}")


# ------------------------------------------------------------
# WIREGUARD
# ------------------------------------------------------------

wg_ns = widgets.Text(
    description="Namespace:"
)

wg_iface = widgets.Text(
    description="Interface:",
    value="wg0"
)

wg_config = widgets.Text(
    description="Config:",
    placeholder="/etc/wireguard/namespace/wg0.conf"
)

wg_enabled = widgets.Checkbox(
    description="Enabled",
    value=True
)

btn_add_wg = widgets.Button(
    description="Add WireGuard",
    button_style="info"
)


def handle_add_wg(button):
    try:
        item = {
            "ns": wg_ns.value.strip(),
            "iface": wg_iface.value.strip(),
            "config": wg_config.value.strip(),
            "enabled": wg_enabled.value
        }

        candidate = json.loads(json.dumps(config))
        candidate["wireguard"].append(item)

        validate_config(candidate)

        config["wireguard"].append(item)

        refresh_topology()

        log(f"WireGuard added: {item['ns']}/{item['iface']}")

    except Exception as error:
        log(f"WIREGUARD ERROR: {error}")


# ------------------------------------------------------------
# PHYSICAL HOST INTERFACES
# ------------------------------------------------------------

physical_ns = widgets.Text(
    description="Namespace:"
)

physical_iface = widgets.Text(
    description="Host NIC:"
)

physical_rename = widgets.Text(
    description="Rename:",
    placeholder="Optional"
)

btn_add_physical = widgets.Button(
    description="Add Physical NIC",
    button_style="warning"
)


def handle_add_physical(button):
    try:
        item = {
            "ns": physical_ns.value.strip(),
            "iface": physical_iface.value.strip(),
            "rename": (
                physical_rename.value.strip()
                or physical_iface.value.strip()
            )
        }

        candidate = json.loads(json.dumps(config))
        candidate["physical"].append(item)

        validate_config(candidate)

        config["physical"].append(item)

        refresh_topology()

        log(
            f"Physical NIC configuration added: "
            f"{item['iface']} -> {item['ns']}"
        )

        log(
            "Reminder: actual movement requires explicit "
            "allowlisting and authorization."
        )

    except Exception as error:
        log(f"PHYSICAL ERROR: {error}")


# ------------------------------------------------------------
# IP ADDRESSING
# ------------------------------------------------------------

ip_ns = widgets.Text(description="Namespace:")
ip_iface = widgets.Text(description="Interface:")

ip_addr = widgets.Text(
    description="IP/Mask:",
    placeholder="192.0.2.10/24"
)

ip_version = widgets.Dropdown(
    options=["IPv4", "IPv6"],
    value="IPv4",
    description="Family:"
)

btn_add_ip = widgets.Button(
    description="Add IP",
    button_style="info"
)


def handle_add_ip(button):
    try:
        item = {
            "ns": ip_ns.value.strip(),
            "iface": ip_iface.value.strip(),
            "addr": ip_addr.value.strip(),
            "version": ip_version.value
        }

        candidate = json.loads(json.dumps(config))
        candidate["ips"].append(item)

        validate_config(candidate)

        config["ips"].append(item)

        log(
            f"IP added: {item['ns']}/{item['iface']} "
            f"{item['addr']}"
        )

    except Exception as error:
        log(f"IP ERROR: {error}")


# ------------------------------------------------------------
# STATIC ROUTING
# ------------------------------------------------------------

route_ns = widgets.Text(description="Namespace:")

route_dst = widgets.Text(
    description="Destination:",
    placeholder="0.0.0.0/0"
)

route_via = widgets.Text(
    description="Gateway:",
    placeholder="192.0.2.1"
)

route_dev = widgets.Text(
    description="Interface:",
    placeholder="eth0"
)

route_table = widgets.Text(
    description="Table:",
    value="main"
)

route_metric = widgets.IntText(
    description="Metric:",
    value=100
)

route_family = widgets.Dropdown(
    options=["IPv4", "IPv6"],
    value="IPv4",
    description="Family:"
)

route_onlink = widgets.Checkbox(
    description="On-link",
    value=False
)

btn_add_route = widgets.Button(
    description="Add Route",
    button_style="info"
)


def handle_add_route(button):
    try:
        item = {
            "ns": route_ns.value.strip(),
            "dst": route_dst.value.strip(),
            "family": route_family.value,
            "table": route_table.value.strip() or "main",
            "metric": route_metric.value,
            "onlink": route_onlink.value
        }

        if route_via.value.strip():
            item["via"] = route_via.value.strip()

        if route_dev.value.strip():
            item["dev"] = route_dev.value.strip()

        candidate = json.loads(json.dumps(config))
        candidate["routes"].append(item)

        validate_config(candidate)

        config["routes"].append(item)

        log(
            f"Route added: {item['ns']} "
            f"{item['dst']}"
        )

    except Exception as error:
        log(f"ROUTE ERROR: {error}")


# ------------------------------------------------------------
# POLICY ROUTING
# ------------------------------------------------------------

rule_ns = widgets.Text(description="Namespace:")

rule_source = widgets.Text(
    description="Source:",
    placeholder="172.16.1.0/24"
)

rule_destination = widgets.Text(
    description="Destination:",
    placeholder="0.0.0.0/0"
)

rule_priority = widgets.IntText(
    description="Priority:",
    value=100
)

rule_table = widgets.IntText(
    description="Table ID:",
    value=100
)

rule_family = widgets.Dropdown(
    options=["IPv4", "IPv6"],
    value="IPv4",
    description="Family:"
)

btn_add_rule = widgets.Button(
    description="Add Rule",
    button_style="info"
)


def handle_add_rule(button):
    try:
        item = {
            "ns": rule_ns.value.strip(),
            "priority": rule_priority.value,
            "table": rule_table.value,
            "family": rule_family.value
        }

        if rule_source.value.strip():
            item["from"] = rule_source.value.strip()

        if rule_destination.value.strip():
            item["to"] = rule_destination.value.strip()

        candidate = json.loads(json.dumps(config))
        candidate["rules"].append(item)

        validate_config(candidate)

        config["rules"].append(item)

        log(f"Policy rule added: {item}")

    except Exception as error:
        log(f"RULE ERROR: {error}")


# ------------------------------------------------------------
# IP FORWARDING
# ------------------------------------------------------------

forward_ns = widgets.Text(description="Namespace:")

forward_ipv4 = widgets.Checkbox(
    description="IPv4 forwarding",
    value=False
)

forward_ipv6 = widgets.Checkbox(
    description="IPv6 forwarding",
    value=False
)

btn_add_forwarding = widgets.Button(
    description="Set Forwarding",
    button_style="info"
)


def handle_add_forwarding(button):
    try:
        item = {
            "ns": forward_ns.value.strip(),
            "ipv4": forward_ipv4.value,
            "ipv6": forward_ipv6.value
        }

        candidate = json.loads(json.dumps(config))

        candidate["forwarding"] = [
            x for x in candidate["forwarding"]
            if x["ns"] != item["ns"]
        ]

        candidate["forwarding"].append(item)

        validate_config(candidate)

        config["forwarding"] = candidate["forwarding"]

        log(f"Forwarding configuration set: {item}")

    except Exception as error:
        log(f"FORWARDING ERROR: {error}")


In [ ]:

# ============================================================
# CELL 10: JSON IMPORT / EXPORT
# ============================================================

file_upload = widgets.FileUpload(
    description="Import JSON",
    accept=".json",
    multiple=False
)

btn_export = widgets.Button(
    description="Export JSON",
    button_style="warning"
)

btn_validate = widgets.Button(
    description="Validate",
    button_style="info"
)

btn_apply = widgets.Button(
    description="Apply / Deploy",
    button_style="success",
    layout=widgets.Layout(
        width="180px",
        height="42px"
    )
)

btn_cleanup = widgets.Button(
    description="Cleanup",
    button_style="danger",
    layout=widgets.Layout(
        width="180px",
        height="42px"
    )
)

btn_clear = widgets.Button(
    description="Clear Configuration",
    button_style="warning"
)


def extract_uploaded_file(upload):
    """
    Supports ipywidgets 7 and 8 FileUpload formats.
    """

    value = upload.value

    if not value:
        raise ValueError("No file selected")

    if isinstance(value, dict):
        entry = next(iter(value.values()))

    elif isinstance(value, (tuple, list)):
        entry = value[0]

    else:
        raise ValueError("Unsupported upload format")

    content = entry.get("content")

    if content is None:
        raise ValueError("Uploaded file has no content")

    if hasattr(content, "tobytes"):
        content = content.tobytes()

    return bytes(content)


def import_configuration(data):
    global config

    candidate = json.loads(
        json.dumps(data)
    )

    validate_config(candidate)

    # Replace only after successful validation.
    config = candidate

    refresh_topology()

    log("Configuration imported successfully")


def handle_file_upload(change):
    try:
        raw = extract_uploaded_file(file_upload)

        imported = json.loads(
            raw.decode("utf-8")
        )

        import_configuration(imported)

    except Exception as error:
        log(f"IMPORT ERROR: {error}")


def handle_export(button):
    try:
        validate_config(config)

        filename = "network_topology_v2.json"

        Path(filename).write_text(
            json.dumps(config, indent=2),
            encoding="utf-8"
        )

        log(f"Configuration exported: {filename}")

        with output_log:
            display(HTML(
                f'<a href="{filename}" download>'
                f'Download {filename}</a>'
            ))

    except Exception as error:
        log(f"EXPORT ERROR: {error}")


def handle_validate(button):
    try:
        validate_config(config)
        log("VALIDATION PASSED")

    except Exception as error:
        log(f"VALIDATION FAILED: {error}")


def handle_apply(button):
    try:
        deploy_configuration(config)

    except Exception as error:
        log(f"DEPLOYMENT ERROR: {error}")


def handle_cleanup(button):
    try:
        cleanup_configuration()

    except Exception as error:
        log(f"CLEANUP ERROR: {error}")


def handle_clear(button):
    global config

    if deployment_active:
        log("Cannot clear configuration during deployment")
        return

    config = json.loads(
        json.dumps(EMPTY_CONFIG)
    )

    refresh_topology()

    log("Configuration cleared")


def refresh_topology():
    app.update_graph(config)


# Bind events.
btn_add_node.on_click(handle_add_node)
btn_add_link.on_click(handle_add_link)
btn_add_switch.on_click(handle_add_switch)
btn_add_wg.on_click(handle_add_wg)
btn_add_physical.on_click(handle_add_physical)
btn_add_ip.on_click(handle_add_ip)
btn_add_route.on_click(handle_add_route)
btn_add_rule.on_click(handle_add_rule)
btn_add_forwarding.on_click(handle_add_forwarding)

btn_export.on_click(handle_export)
btn_validate.on_click(handle_validate)
btn_apply.on_click(handle_apply)
btn_cleanup.on_click(handle_cleanup)
btn_clear.on_click(handle_clear)

file_upload.observe(
    handle_file_upload,
    names="value"
)

print("Configuration management initialized")


In [ ]:

# ============================================================
# CELL 12: FINAL GUI ASSEMBLY
# ============================================================

# ------------------------------------------------------------
# DEPLOYMENT SETTINGS
# ------------------------------------------------------------

dry_run_control = widgets.Checkbox(
    description="Dry-run mode (no changes)",
    value=True,
    indent=False
)

physical_control = widgets.Checkbox(
    description="Allow physical NIC movement",
    value=False,
    indent=False
)

physical_allowlist_input = widgets.Text(
    description="Allowed NICs:",
    placeholder="Only explicitly authorized interface names"
)


def handle_dry_run(change):
    global DRY_RUN

    if change["name"] == "value":
        DRY_RUN = change["new"]
        log(f"Dry-run mode: {DRY_RUN}")


def handle_physical_control(change):
    global ALLOW_PHYSICAL_MOVEMENT

    if change["name"] == "value":
        ALLOW_PHYSICAL_MOVEMENT = change["new"]

        log(
            f"Physical interface movement enabled: "
            f"{ALLOW_PHYSICAL_MOVEMENT}"
        )


def handle_physical_allowlist(change):
    global PHYSICAL_INTERFACE_ALLOWLIST

    if change["name"] == "value":
        PHYSICAL_INTERFACE_ALLOWLIST = {
            x.strip()
            for x in change["new"].split(",")
            if x.strip()
        }

        log(
            "Physical interface allowlist updated: "
            + str(sorted(PHYSICAL_INTERFACE_ALLOWLIST))
        )


dry_run_control.observe(
    handle_dry_run,
    names="value"
)

physical_control.observe(
    handle_physical_control,
    names="value"
)

physical_allowlist_input.observe(
    handle_physical_allowlist,
    names="value"
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

topology_view = (
    app.cytoscape_widget
    if CYTOSCAPE_AVAILABLE
    else widgets.HTML(
        "<p>Cytoscape unavailable. "
        "Install ipycytoscape for the interactive graph.</p>"
    )
)

interface = widgets.VBox([

    widgets.HTML(
        "<h2>Network Namespace & Topology Builder</h2>"
        "<p>Virtual networking, bridges, WireGuard, "
        "routing and optional physical interfaces.</p>"
    ),

    widgets.HTML("<h3>Live Topology Preview</h3>"),

    topology_view,

    widgets.HTML("<hr><h3>1. Import Configuration</h3>"),

    file_upload,

    widgets.HTML("<hr><h3>2. Build Namespaces</h3>"),

    widgets.HBox([
        ns_input,
        btn_add_node
    ]),

    widgets.HTML("<hr><h3>3. Build Veth Links</h3>"),

    widgets.VBox([
        widgets.HBox([
            link_name1,
            link_ns1,
            link_name1_rename
        ]),
        widgets.HBox([
            link_name2,
            link_ns2,
            link_name2_rename
        ]),
        btn_add_link
    ]),

    widgets.HTML("<hr><h3>4. Configure Dumb Switches</h3>"),

    widgets.VBox([
        widgets.HBox([
            switch_ns,
            switch_bridge
        ]),
        widgets.HBox([
            switch_ports,
            switch_stp,
            switch_vlan
        ]),
        btn_add_switch
    ]),

    widgets.HTML("<hr><h3>5. Configure WireGuard</h3>"),

    widgets.VBox([
        widgets.HBox([
            wg_ns,
            wg_iface
        ]),
        widgets.HBox([
            wg_config,
            wg_enabled
        ]),
        btn_add_wg
    ]),

    widgets.HTML(
        "<hr><h3>6. Optional Physical Host Interfaces</h3>"
    ),

    widgets.VBox([
        widgets.HBox([
            physical_ns,
            physical_iface,
            physical_rename
        ]),
        btn_add_physical
    ]),

    widgets.HTML("<hr><h3>7. Assign IPv4 / IPv6 Addresses</h3>"),

    widgets.VBox([
        widgets.HBox([
            ip_ns,
            ip_iface
        ]),
        widgets.HBox([
            ip_addr,
            ip_version,
            btn_add_ip
        ])
    ]),

    widgets.HTML("<hr><h3>8. Static Routing</h3>"),

    widgets.VBox([
        widgets.HBox([
            route_ns,
            route_family
        ]),
        widgets.HBox([
            route_dst,
            route_via
        ]),
        widgets.HBox([
            route_dev,
            route_table,
            route_metric
        ]),
        widgets.HBox([
            route_onlink,
            btn_add_route
        ])
    ]),

    widgets.HTML("<hr><h3>9. Policy Routing</h3>"),

    widgets.VBox([
        widgets.HBox([
            rule_ns,
            rule_family
        ]),
        widgets.HBox([
            rule_source,
            rule_destination
        ]),
        widgets.HBox([
            rule_priority,
            rule_table
        ]),
        btn_add_rule
    ]),

    widgets.HTML("<hr><h3>10. IP Forwarding</h3>"),

    widgets.VBox([
        forward_ns,
        widgets.HBox([
            forward_ipv4,
            forward_ipv6
        ]),
        btn_add_forwarding
    ]),

    widgets.HTML("<hr><h3>11. Deployment Settings</h3>"),

    widgets.VBox([
        dry_run_control,
        physical_control,
        physical_allowlist_input
    ]),

    widgets.HTML("<hr><h3>12. Master Controls</h3>"),

    widgets.HBox([
        btn_validate,
        btn_apply,
        btn_export,
        btn_cleanup
    ]),

    btn_clear,

    widgets.HTML("<hr><h3>Execution Console</h3>"),

    output_log

], layout=widgets.Layout(
    width="100%",
    gap="10px"
))


display(interface)

print("Network Namespace Builder GUI ready")


In [ ]:

# ============================================================
# CELL 13: FINAL CONFIGURATION AND DRY-RUN TEST
# ============================================================

def run_predeployment_test():
    global DRY_RUN

    previous_dry_run = DRY_RUN

    try:
        DRY_RUN = True
        dry_run_control.value = True

        log("========== PRE-DEPLOYMENT TEST ==========")

        # Test JSON serialization.
        serialized = json.dumps(config, indent=2)
        restored = json.loads(serialized)

        log("PASS: JSON serialization")

        # Test configuration validation.
        validate_config(restored)

        log("PASS: Configuration validation")

        # Check namespace count.
        log(
            f"PASS: {len(config['namespaces'])} namespaces"
        )

        # Check virtual links.
        log(
            f"PASS: {len(config['links'])} veth connections"
        )

        # Check switches.
        log(
            f"PASS: {len(config['switches'])} bridges"
        )

        # Check WireGuard.
        enabled_wg = sum(
            1 for x in config["wireguard"]
            if x.get("enabled", True)
        )

        log(
            f"WireGuard interfaces enabled: {enabled_wg}"
        )

        # Check physical interfaces.
        log(
            f"Physical interfaces configured: "
            f"{len(config['physical'])}"
        )

        # Check routing.
        log(
            f"PASS: {len(config['routes'])} static routes"
        )

        log(
            f"PASS: {len(config['rules'])} policy rules"
        )

        log(
            f"PASS: {len(config['forwarding'])} forwarding "
            "configurations"
        )

        # Execute deployment in dry-run mode.
        log("Starting dry-run deployment")

        deploy_configuration(config)

        log("PASS: Dry-run deployment completed")

        log("========== TEST COMPLETE ==========")

    except Exception as error:
        log(f"PRE-DEPLOYMENT TEST FAILED: {error}")
        raise

    finally:
        DRY_RUN = previous_dry_run
        dry_run_control.value = previous_dry_run


run_test_button = widgets.Button(
    description="Run Pre-deployment Test",
    button_style="info"
)

run_test_button.on_click(
    lambda button: run_predeployment_test()
)

display(run_test_button)


In [ ]:

# ============================================================
# CELL 14: NETWORK DIAGNOSTICS
# ============================================================

def inspect_namespace(ns):
    validate_name(ns, "Namespace")

    if DRY_RUN:
        raise RuntimeError(
            "Diagnostics require a real deployment"
        )

    if not namespace_exists(ns):
        raise RuntimeError(
            f"Namespace does not exist: {ns}"
        )

    log(f"========== INSPECTING {ns} ==========")

    execute(
        ["ip", "-br", "link"],
        namespace=ns
    )

    execute(
        ["ip", "-br", "address"],
        namespace=ns
    )

    execute(
        ["ip", "-4", "route", "show", "table", "all"],
        namespace=ns
    )

    execute(
        ["ip", "-6", "route", "show", "table", "all"],
        namespace=ns
    )

    execute(
        ["ip", "-4", "rule", "show"],
        namespace=ns
    )

    execute(
        ["ip", "-6", "rule", "show"],
        namespace=ns
    )


def inspect_all_namespaces():
    for ns in config["namespaces"]:
        inspect_namespace(ns)


def ping_from_namespace(ns, destination, count=3):
    validate_name(ns, "Namespace")

    address = ipaddress.ip_address(destination)

    if DRY_RUN:
        raise RuntimeError(
            "Ping requires real deployment"
        )

    family = "-4" if address.version == 4 else "-6"

    execute([
        "ping",
        family,
        "-c", str(count),
        "-W", "2",
        str(address)
    ], namespace=ns)


print("Diagnostics functions available:")
print("  inspect_namespace('client1')")
print("  inspect_all_namespaces()")
print("  ping_from_namespace('client1', '198.51.100.1')")
